In [4]:
queries = ["枚举","递归","分治","二分查找","归并排序","快速排序","堆排序",
            "插入排序","冒泡排序","选择排序","计数排序","桶排序",
            "基数排序","滑动窗口","双指针","前缀和","贪心算法","Kruskal",
            "Prim","最小区间覆盖","Dijkstra","Huffman编码","作业调度",
            "最小字典序构造","动态规划","最长公共子序列","背包问题","编辑距离",
            "最近点对","DFS","BFS","图连通块","拓扑排序","最短路径","最小步数",
            "回溯","N皇后","图着色","剪枝","最小生成树","网络流","KMP","局部搜索",
            "模拟退火","遗传算法"]

In [6]:
import requests
import json

LLM_MODEL = "Qwen/Qwen3-32B"
BASE_URL = "https://api.siliconflow.cn/v1/chat/completions"
API_KEY = "sk-vhfwhragmjtribovcmvkvfyyrqvargpqdeqwqcfllateeqtz"

# 构造问题列表
queries_pre = [
    f"{q}的前置知识有哪些？请只返回 Python 列表，例如：['A','B','C']"
    for q in queries
]

# 请求头
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

results = {}

for full_question in queries_pre:
    # 提取算法名称（去除固定问句）
    algo_name = full_question.replace("的前置知识有哪些？请只返回 Python 列表，例如：['A','B','C']", "")

    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": "你是一个算法学习导师，只返回 Python 列表格式的结果。"},
            {"role": "user", "content": full_question}
        ],
        "temperature": 0.3
    }

    response = requests.post(BASE_URL, headers=headers, data=json.dumps(payload))
    data = response.json()

    # 防止 KeyError：先检查响应格式
    content = data.get("choices", [{}])[0].get("message", {}).get("content", "")

    # 清洗文本
    answer = content.strip()

    # 尝试解析 Python 列表，否则包成单元素列表
    try:
        prereq_list = eval(answer)
        if not isinstance(prereq_list, list):
            prereq_list = [answer]
    except:
        prereq_list = [answer]

    # 存成 {算法名称：前置知识列表}
    results[algo_name] = prereq_list


output_path = "algorithm_prerequisites.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"前置知识字典已保存到：{output_path}")

# 同时输出到屏幕
print(json.dumps(results, ensure_ascii=False, indent=2))

前置知识字典已保存到：algorithm_prerequisites.json
{
  "枚举": [
    "循环结构",
    "条件判断",
    "数据结构（如列表、元组）",
    "函数定义",
    "算法复杂度分析"
  ],
  "递归": [
    "函数定义",
    "栈结构",
    "递归调用",
    "递归终止条件",
    "递归与循环的区别"
  ],
  "分治": [
    "递归",
    "时间复杂度分析",
    "问题分解能力",
    "合并子问题结果的方法"
  ],
  "二分查找": [
    "有序数组",
    "递归或循环",
    "时间复杂度分析"
  ],
  "归并排序": [
    "分治算法",
    "递归",
    "数组操作",
    "时间复杂度分析",
    "空间复杂度分析"
  ],
  "快速排序": [
    "递归",
    "分治策略",
    "数组操作",
    "比较和交换元素",
    "基准值选择"
  ],
  "堆排序": [
    "数据结构基础",
    "二叉树",
    "完全二叉树",
    "堆（最大堆/最小堆）",
    "递归",
    "数组操作"
  ],
  "插入排序": [
    "循环结构",
    "条件判断",
    "数组操作",
    "时间复杂度",
    "空间复杂度"
  ],
  "冒泡排序": [
    "循环结构",
    "条件判断",
    "列表操作",
    "比较运算",
    "算法复杂度"
  ],
  "选择排序": [
    "循环结构",
    "比较操作",
    "交换操作",
    "数组/列表基础"
  ],
  "计数排序": [
    "数组操作",
    "循环结构",
    "条件判断",
    "理解排序算法的基本概念",
    "了解时间与空间复杂度"
  ],
  "桶排序": [
    "数组操作",
    "循环结构",
    "条件判断",
    "函数定义",
    "列表推导式",
    "排序算法基础概念",
    "均匀分布与数据范围"
 

In [8]:
import json
import numpy as np
import faiss
import ollama

# ================================================================
# 1. 加载 chunks（新的格式：一个 list，每个元素是一个 chunk 对象）
# ================================================================
with open("refined_document_chunks.json", "r", encoding="utf-8") as f:
    chunk_list = json.load(f)

# 提取 chunk_id → content
chunk_ids = [str(item["chunk_id"]) for item in chunk_list]
chunk_texts = [item["content"] for item in chunk_list]

print(f"已加载 {len(chunk_ids)} 个 chunk。")

# ================================================================
# 2. 加载前置知识列表 JSON
# ================================================================
with open("algorithm_prerequisites.json", "r", encoding="utf-8") as f:
    prerequisites = json.load(f)

# ================================================================
# 3. embedding 函数：调用本地 bge-m3
# ================================================================
def embed(text: str):
    """调用本地 bge-m3 模型获取文本 embedding"""
    resp = ollama.embeddings(
        model="bge-m3",
        prompt=text
    )
    return np.array(resp["embedding"], dtype="float32")


# ================================================================
# 4. 构建 chunk 向量库
# ================================================================
print("⚙️ 正在构建 chunk embedding 向量库 ... 这可能需要几秒...")

chunk_embeddings = np.array([embed(text) for text in chunk_texts], dtype="float32")
embedding_dim = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(embedding_dim)
index.add(chunk_embeddings)

print("✅ 向量库构建完成！")


# ================================================================
# 5. 语义搜索函数
# ================================================================
def semantic_search(query: str, top_k=1):
    """返回语义最相近的 chunk_id"""
    q_emb = embed(query).reshape(1, -1)
    distances, indices = index.search(q_emb, top_k)
    best_idx = indices[0][0]
    return chunk_ids[best_idx]


# ================================================================
# 6. 主逻辑：为每个前置知识找最相关的 chunk
# ================================================================
mapping = {}

for algo, prereq_list in prerequisites.items():
    mapping[algo] = {}

    for pre in prereq_list:
        try:
            chunk_id = semantic_search(pre)
        except:
            chunk_id = "null"
        mapping[algo][pre] = chunk_id


# ================================================================
# 7. 输出 & 保存
# ================================================================
output_path = "algo_chunk_mapping1.json"

print("\n🎉 匹配完成！最终结果如下：")
print(json.dumps(mapping, ensure_ascii=False, indent=2))

with open(output_path, "w", encoding="utf-8") as fw:
    json.dump(mapping, fw, ensure_ascii=False, indent=2)

print(f"\n✨ 已保存至 {output_path}")


已加载 11587 个 chunk。
⚙️ 正在构建 chunk embedding 向量库 ... 这可能需要几秒...
✅ 向量库构建完成！

🎉 匹配完成！最终结果如下：
{
  "枚举": {
    "循环结构": "7762",
    "条件判断": "8868",
    "数据结构（如列表、元组）": "897",
    "函数定义": "7631",
    "算法复杂度分析": "7750"
  },
  "递归": {
    "函数定义": "7631",
    "栈结构": "10247",
    "递归调用": "10223",
    "递归终止条件": "10223",
    "递归与循环的区别": "10223"
  },
  "分治": {
    "递归": "10223",
    "时间复杂度分析": "7808",
    "问题分解能力": "8880",
    "合并子问题结果的方法": "8672"
  },
  "二分查找": {
    "有序数组": "11238",
    "递归或循环": "10223",
    "时间复杂度分析": "7808"
  },
  "归并排序": {
    "分治算法": "8650",
    "递归": "10223",
    "数组操作": "7941",
    "时间复杂度分析": "7808",
    "空间复杂度分析": "7853"
  },
  "快速排序": {
    "递归": "10223",
    "分治策略": "5562",
    "数组操作": "7941",
    "比较和交换元素": "723",
    "基准值选择": "9506"
  },
  "堆排序": {
    "数据结构基础": "9176",
    "二叉树": "8172",
    "完全二叉树": "8176",
    "堆（最大堆/最小堆）": "583",
    "递归": "10223",
    "数组操作": "7941"
  },
  "插入排序": {
    "循环结构": "7762",
    "条件判断": "8868",
    "数组操作": "7941",
    "时间复杂度": "7808",
   

In [ ]:

# 1. 读入 JSON 文件
with open("algo_chunk_mapping1.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# 2. 构建知识图谱
graph = []

for start_concept, prereqs in data.items():
    # prereqs 是一个 dict，例如：
    # {"循环结构": "7762", "条件判断": "8868", ...}
    for prereq_name, chunk_id in prereqs.items():
        graph.append({
            "start_node": {
                "properties": {
                    "name": start_concept,
                    "chunk id":chunk_id
                }
            },
            "relation": "前置知识",
            "end_node": {
                "properties": {
                    "name": prereq_name,
                    "chunk id": chunk_id
                }
            }
        })

# 3. 输出到 JSON 文件
with open("knowledge_graph.json", "w", encoding="utf-8") as f:
    json.dump(graph, f, ensure_ascii=False, indent=2)

graph[:3]  # 显示前 3 条检查


[{'start_node': {'properties': {'name': '枚举'}},
  'relation': '前置知识',
  'end_node': {'properties': {'name': '循环结构', 'chunk id': '7762'}}},
 {'start_node': {'properties': {'name': '枚举'}},
  'relation': '前置知识',
  'end_node': {'properties': {'name': '条件判断', 'chunk id': '8868'}}},
 {'start_node': {'properties': {'name': '枚举'}},
  'relation': '前置知识',
  'end_node': {'properties': {'name': '数据结构（如列表、元组）', 'chunk id': '897'}}}]